DATA GENERATION

In [ ]:
import numpy as np
import pandas as pd
import random
from tqdm import tqdm

np.random.seed(42)
random.seed(42)

# -----------------------------
# CONFIG
# -----------------------------
NUM_USERS = 10000
NUM_RESTAURANTS = 200
NUM_ITEMS = 1500
NUM_SESSIONS = 40000

cities = ["Delhi", "Mumbai", "Bangalore", "Hyderabad", "Chennai"]
cuisines = ["North Indian", "South Indian", "Chinese", "Italian", "Fast Food"]
categories = ["main", "side", "dessert", "drink"]

# -----------------------------
# 1️ Generate Users
# -----------------------------
users = []

for u in range(NUM_USERS):
    segment = np.random.choice(["budget", "premium", "veg_only", "late_night"])
    veg_ratio = 1.0 if segment == "veg_only" else np.random.uniform(0.3, 0.8)
    avg_cart_value = np.random.normal(300 if segment=="budget" else 600, 50)
    users.append([u, segment, veg_ratio, avg_cart_value])

users_df = pd.DataFrame(users, columns=[
    "user_id", "segment", "veg_ratio", "avg_cart_value"
])

# -----------------------------
# 2️ Generate Restaurants
# -----------------------------
restaurants = []

for r in range(NUM_RESTAURANTS):
    cuisine = np.random.choice(cuisines)
    city = np.random.choice(cities)
    price_range = np.random.choice([1,2,3])  # 1=low, 3=premium
    restaurants.append([r, cuisine, city, price_range])

restaurants_df = pd.DataFrame(restaurants, columns=[
    "restaurant_id", "cuisine", "city", "price_range"
])

# -----------------------------
# 3️ Generate Items
# -----------------------------
items = []

for i in range(NUM_ITEMS):
    cuisine = np.random.choice(cuisines)
    category = np.random.choice(categories, p=[0.4,0.25,0.2,0.15])
    price = np.random.randint(50, 400)
    veg_flag = np.random.choice([0,1], p=[0.3,0.7])
    popularity = np.random.uniform(0,1)
    items.append([i, cuisine, category, price, veg_flag, popularity])

items_df = pd.DataFrame(items, columns=[
    "item_id","cuisine","category","price","veg_flag","popularity"
])

# -----------------------------
# 4️ Meal Templates
# -----------------------------
meal_templates = {
    "biryani_meal": ["main", "side", "dessert", "drink"],
    "pizza_meal": ["main", "side", "drink"],
    "dosa_meal": ["main", "side", "drink"],
    "burger_meal": ["main", "side", "drink"]
}

# -----------------------------
# 5️ Generate Sessions
# -----------------------------
sessions = []

for s in tqdm(range(NUM_SESSIONS)):

    user = users_df.sample(1).iloc[0]
    restaurant = restaurants_df.sample(1).iloc[0]

    hour = np.random.randint(0,24)
    weekend = np.random.choice([0,1], p=[0.7,0.3])

    meal_type = np.random.choice(list(meal_templates.keys()))
    structure = meal_templates[meal_type]

    cart_items = []

    for step, category in enumerate(structure):

        # Filter items for restaurant cuisine
        candidate_pool = items_df[
            (items_df["category"]==category)
        ]

        if user["segment"]=="veg_only":
            candidate_pool = candidate_pool[candidate_pool["veg_flag"]==1]

        if len(candidate_pool)==0:
            continue

        item = candidate_pool.sample(1).iloc[0]

        # Acceptance probability logic
        base_prob = 0.6 if category=="main" else 0.4

        # Time effect
        if 12<=hour<=14 or 19<=hour<=22:
            base_prob += 0.1

        # Weekend dessert boost
        if weekend==1 and category=="dessert":
            base_prob += 0.1

        accept = np.random.rand() < base_prob

        if accept:
            cart_items.append(item["item_id"])

        # Create ranking rows
        sessions.append([
            s,
            user["user_id"],
            restaurant["restaurant_id"],
            hour,
            weekend,
            len(cart_items),
            item["item_id"],
            int(accept)
        ])

sessions_df = pd.DataFrame(sessions, columns=[
    "session_id",
    "user_id",
    "restaurant_id",
    "hour",
    "weekend",
    "cart_size",
    "candidate_item",
    "label"
])

print("Dataset created.")
print("Rows:", len(sessions_df))

100%|██████████| 40000/40000 [01:46<00:00, 374.32it/s]


Dataset created.
Rows: 130024


Merge Code

In [ ]:
# Merge user features
data = sessions_df.merge(users_df, on="user_id", how="left")

# Merge restaurant features
data = data.merge(restaurants_df, on="restaurant_id", how="left")

# Merge item features
data = data.merge(items_df, left_on="candidate_item", right_on="item_id", how="left")

print("Merged shape:", data.shape)
data.head()

Merged shape: (130024, 20)


,session_id,user_id,restaurant_id,hour,weekend,cart_size,candidate_item,label,segment,veg_ratio,avg_cart_value,cuisine_x,city,price_range,item_id,cuisine_y,category,price,veg_flag,popularity
0,0,5779,49,22,0,0,743,0,budget,0.329990,385.827033,Italian,Delhi,1,743,Italian,main,398,1,0.024904
1,0,5779,49,22,0,0,49,0,budget,0.329990,385.827033,Italian,Delhi,1,49,Chinese,side,341,1,0.544054
2,0,5779,49,22,0,1,213,1,budget,0.329990,385.827033,Italian,Delhi,1,213,North Indian,drink,251,1,0.871222
3,1,8469,24,1,0,1,1316,1,premium,0.416069,645.850077,Chinese,Delhi,1,1316,Italian,main,344,1,0.150842
4,1,8469,24,1,0,2,200,1,premium,0.416069,645.850077,Chinese,Delhi,1,200,North Indian,side,384,1,0.027228


Time Buckets

In [ ]:
def get_meal_time(hour):
    if 6 <= hour <= 10:
        return "breakfast"
    elif 11 <= hour <= 15:
        return "lunch"
    elif 16 <= hour <= 18:
        return "snacks"
    elif 19 <= hour <= 23:
        return "dinner"
    else:
        return "late_night"

data["meal_time"] = data["hour"].apply(get_meal_time)

Encode Categorical Variables

In [ ]:
categorical_cols = [
    "segment",
    "cuisine_x",  # restaurant cuisine
    "city",
    "cuisine_y",  # item cuisine
    "category",
    "meal_time"
]

data = pd.get_dummies(data, columns=categorical_cols, drop_first=True)

Train-Test Split

In [ ]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(data, test_size=0.2, random_state=42)

X_train = train.drop(columns=["label","session_id","candidate_item","item_id"])
y_train = train["label"]

X_test = test.drop(columns=["label","session_id","candidate_item","item_id"])
y_test = test["label"]

Recreate Dessert Column

In [ ]:
data["category_dessert"] = (
    (data["category_drink"] == 0) &
    (data["category_main"] == 0) &
    (data["category_side"] == 0)
).astype(int)

Strong Cart Intelligence Features

In [ ]:
# Complementarity flags
data["drink_boost_flag"] = (
    (data["cart_size"] == 1) & (data["category_drink"] == 1)
).astype(int)

data["dessert_boost_flag"] = (
    (data["cart_size"] <= 2) & (data["category_dessert"] == 1)
).astype(int)

# Price sensitivity
data["price_sensitivity"] = data["price"] / (data["avg_cart_value"] + 1)

# Popularity interaction
data["popularity_x_cart"] = data["popularity"] * data["cart_size"]

Redo Split

In [ ]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(data, test_size=0.2, random_state=42)

X_train = train.drop(columns=["label","session_id","candidate_item","item_id"])
y_train = train["label"]

X_test = test.drop(columns=["label","session_id","candidate_item","item_id"])
y_test = test["label"]

Clean Reset Sequence

In [ ]:
from sklearn.model_selection import train_test_split

# Split AFTER adding new features
train, test = train_test_split(data, test_size=0.2, random_state=42)

X_train = train.drop(columns=["label","session_id","candidate_item","item_id"])
y_train = train["label"]

X_test = test.drop(columns=["label","session_id","candidate_item","item_id"])
y_test = test["label"]

Retrain Model

In [ ]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=150,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    use_label_encoder=False
)

model.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [09:25:31] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=150, n_jobs=None,
              num_parallel_tree=None, ...)

In [ ]:
test["pred_score"] = model.predict_proba(X_test)[:,1]

In [ ]:
test["final_score"] = test["pred_score"]

test.loc[
    (test["cart_size"] == 1) &
    (test["category_drink"] == 1),
    "final_score"
] += 0.05

test.loc[
    (test["weekend"] == 1) &
    (test["category_dessert"] == 1),
    "final_score"
] += 0.03

EVALUATION CODE BLOCK

In [ ]:
from sklearn.metrics import roc_auc_score
import numpy as np

test = test.copy()

# Predict
test["pred_score"] = model.predict_proba(X_test)[:, 1]
test["final_score"] = test["pred_score"]

# Business boost
test.loc[
    (test["cart_size"] == 1) &
    (test["category_drink"] == 1),
    "final_score"
] += 0.05

if "category_dessert" in test.columns:
    test.loc[
        (test["weekend"] == 1) &
        (test["category_dessert"] == 1),
        "final_score"
    ] += 0.03

# -----------------------------
# AUC
# -----------------------------
auc = roc_auc_score(y_test, test["pred_score"])

# -----------------------------
# Precision@K (Dynamic K)
# -----------------------------
precision_scores = []

for session_id in test["session_id"].unique():
    session_data = test[test["session_id"] == session_id]

    if len(session_data) == 0:
        continue

    K = min(10, len(session_data))

    session_sorted = session_data.sort_values("final_score", ascending=False).head(K)

    precision = session_sorted["label"].sum() / K
    precision_scores.append(precision)

precision_at_k = np.mean(precision_scores)

# -----------------------------
# Business Impact
# -----------------------------
baseline_attach_rate = 0.08
model_attach_rate = precision_at_k
avg_addon_value = 70

aov_lift = (model_attach_rate - baseline_attach_rate) * avg_addon_value

print("New AUC:", round(auc, 4))
print("New Precision@K:", round(precision_at_k, 4))
print("New Projected AOV Lift per Order (₹):", round(aov_lift, 2))


New AUC: 0.9355
New Precision@K: 0.4951
New Projected AOV Lift per Order (₹): 29.06
